## 5. Modelado y evaluación

In [16]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from pathlib import Path

from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from tensorflow.keras import layers, Model
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, KFold

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve
)

import optuna


# Paths
X_train_in = Path("../data/modeling/supervised/X_train.parquet")
X_test_in = Path("../data/modeling/supervised/X_test.parquet")
y_train_in = Path("../data/modeling/supervised/y_train.parquet")
y_test_in = Path("../data/modeling/supervised/y_test.parquet")

X_train_N_in = Path("../data/modeling/unsupervised/X_train_N.parquet")
X_test_N_in = Path("../data/modeling/unsupervised/X_test_N.parquet")
y_test_N_in = Path("../data/modeling/unsupervised/y_test_N.parquet")

#### 1) Revisión general

Cargo los diferentes conjuntos preparados para usar en los diferentes modelos.

In [15]:
X_train = pd.read_parquet(X_train_in)
X_test = pd.read_parquet(X_test_in)
y_train = pd.read_parquet(y_train_in)
y_test = pd.read_parquet(y_test_in)

X_train_N = pd.read_parquet(X_train_N_in)
X_test_N = pd.read_parquet(X_test_N_in)
y_test_N = pd.read_parquet(y_test_N_in)

Soluciono un par de errores de tipo pd a np: 

In [3]:
y_train = y_train.squeeze().to_numpy()
y_test = y_test.squeeze().to_numpy()
X_train = X_train.squeeze().to_numpy()
X_test = X_test.squeeze().to_numpy()

In [19]:
X_train_N = X_train_N.squeeze().to_numpy()
X_test_N = X_test_N.squeeze().to_numpy()
y_test_N = y_test_N.squeeze().to_numpy()

#### 2) Modelos Supervisados

Dada la separación hecha en el notebook 4 más la codificación de la variable respuesta en este caso '0' representa Ataque y '1' Benigno.

**XGBoost**

In [4]:
# Weights adjustment
neg, pos = np.bincount(y_train)
scale_pos_weight = neg / pos

print("scale_pos_weight:", scale_pos_weight)

scale_pos_weight: 0.23960760013846696


In [6]:
X = np.array(X_train)
y = np.array(y_train)

cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)

def objective(trial):

    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.1),
        "max_depth": trial.suggest_int("max_depth", 3, 6),  # smaller range
        "subsample": trial.suggest_float("subsample", 0.8, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.8, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 5),
        "gamma": trial.suggest_float("gamma", 0.0, 0.3),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 0.3),
        "reg_lambda": trial.suggest_float("reg_lambda", 1.0, 2.0),
    }

    scores = []

    for train_idx, valid_idx in cv.split(X, y):

        X_tr, X_val = X[train_idx], X[valid_idx]
        y_tr, y_val = y[train_idx], y[valid_idx]

        model = XGBClassifier(

            **params,

            n_estimators=400,   # 🔥 BIG SPEED BOOST (was 2000)

            scale_pos_weight=scale_pos_weight,

            tree_method="hist",
            device="cuda",

            eval_metric="logloss",
            random_state=42,
            n_jobs=-1
        )

        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_val, y_val)],
            verbose=False
        )

        preds = model.predict_proba(X_val)[:, 1]

        scores.append(average_precision_score(y_val, preds))

    return float(np.mean(scores))

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)  # reduce from 30 → 10 for verification

best_params = study.best_params
print(best_params)

[I 2026-05-29 14:10:06,268] A new study created in memory with name: no-name-1e2fb9f0-4975-4eb4-b175-ac37cc6fcfea
/home/adrian/Documentos/TFG/venv/lib/python3.12/site-packages/xgboost/core.py:751: UserWarning: [14:10:33] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
[I 2026-05-29 14:11:13,832] Trial 0 finished with value: 0.9946690389869809 and parameters: {'learning_rate': 0.0350348399443422, 'max_depth': 3, 'subsample': 0.8227058081241172, 'colsample_bytree': 0.8757559547842357, 'min_child_weight': 4, 'gamma': 0.22308041659634903, 'reg_alpha': 0.19638796669832384, 're

{'learning_rate': 0.09890860585819196, 'max_depth': 6, 'subsample': 0.8670642232678142, 'colsample_bytree': 0.9579982000396621, 'min_child_weight': 2, 'gamma': 0.05722614875920762, 'reg_alpha': 0.17154684495225483, 'reg_lambda': 1.9788954698285044}


In [10]:
base_model = XGBClassifier(
    **best_params,
    tree_method="hist",
    device="cuda",
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    n_estimators=2000  # important for early stopping stability
)

# I wanna avoid bias
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=42
)

# Fitting the model
base_model.fit(
    X_tr,
    y_tr,
    eval_set=[(X_val, y_val)],
    verbose=50
)

[0]	validation_0-logloss:0.61887
[50]	validation_0-logloss:0.13050
[100]	validation_0-logloss:0.11905
[150]	validation_0-logloss:0.11457
[200]	validation_0-logloss:0.11211
[250]	validation_0-logloss:0.11032
[300]	validation_0-logloss:0.10891
[350]	validation_0-logloss:0.10785
[400]	validation_0-logloss:0.10687
[450]	validation_0-logloss:0.10610
[500]	validation_0-logloss:0.10537
[550]	validation_0-logloss:0.10483
[600]	validation_0-logloss:0.10435
[650]	validation_0-logloss:0.10392
[700]	validation_0-logloss:0.10346
[750]	validation_0-logloss:0.10306
[800]	validation_0-logloss:0.10268
[850]	validation_0-logloss:0.10237
[900]	validation_0-logloss:0.10205
[950]	validation_0-logloss:0.10180
[1000]	validation_0-logloss:0.10149
[1050]	validation_0-logloss:0.10120
[1100]	validation_0-logloss:0.10097
[1150]	validation_0-logloss:0.10068
[1200]	validation_0-logloss:0.10049
[1250]	validation_0-logloss:0.10028
[1300]	validation_0-logloss:0.10008
[1350]	validation_0-logloss:0.09986
[1400]	validati

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.9579982000396621
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",'cuda'
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets impor

In [11]:
# PREDICTIONS
y_proba = base_model.predict_proba(X_test)[:, 1]

# OPTIMAL THRESHOLD (F1-based)
precision, recall, thresholds = precision_recall_curve(y_test, y_proba)

f1_scores = (2 * precision * recall) / (precision + recall + 1e-9)

# align lengths (thresholds has length N-1)
best_idx = f1_scores[:-1].argmax()
best_threshold = thresholds[best_idx]

y_pred = (y_proba >= best_threshold).astype(int)

# BASE MODEL EVALUATION
print("\nBest threshold:", best_threshold)

print("\n=== CONFUSION MATRIX ===")
print(confusion_matrix(y_test, y_pred))

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred))

print("\nROC-AUC:", roc_auc_score(y_test, y_proba))
print("PR-AUC:", average_precision_score(y_test, y_proba))


Best threshold: 0.20015606

=== CONFUSION MATRIX ===
[[ 73374   9860]
 [   445 346931]]

=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

           0       0.99      0.88      0.93     83234
           1       0.97      1.00      0.99    347376

    accuracy                           0.98    430610
   macro avg       0.98      0.94      0.96    430610
weighted avg       0.98      0.98      0.98    430610


ROC-AUC: 0.9900909101072826
PR-AUC: 0.9974107530031939


In [ ]:
# FEATURE IMPORTANCE
importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": base_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

print("\nTop 20 Features:")
print(importance.head(20))

**Random Forest**

In [13]:
cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)

def objective(trial):

    model = RandomForestClassifier(
        n_estimators=trial.suggest_int("n_estimators", 100, 400),
        max_depth=trial.suggest_int("max_depth", 5, 20),
        min_samples_split=trial.suggest_int("min_samples_split", 2, 15),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 8),
        max_features="sqrt",
        class_weight="balanced",
        n_jobs=-1,
        random_state=42
    )

    scores = []

    for tr, val in cv.split(X_train, y_train):
        model.fit(X_train[tr], y_train[tr])
        preds = model.predict_proba(X_train[val])[:, 1]
        scores.append(average_precision_score(y_train[val], preds))

    return np.mean(scores)


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=5)
best_params = study.best_params
print(best_params)

[I 2026-05-29 14:49:41,889] A new study created in memory with name: no-name-aec6aaf2-13a5-4f31-802e-fe65014c2ca3
[I 2026-05-29 14:54:52,986] Trial 0 finished with value: 0.9966251289615836 and parameters: {'n_estimators': 165, 'max_depth': 18, 'min_samples_split': 8, 'min_samples_leaf': 6, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.9966251289615836.
[I 2026-05-29 14:59:43,617] Trial 1 finished with value: 0.9937688794225414 and parameters: {'n_estimators': 256, 'max_depth': 7, 'min_samples_split': 13, 'min_samples_leaf': 5, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.9966251289615836.
[I 2026-05-29 15:10:35,116] Trial 2 finished with value: 0.9962271474316893 and parameters: {'n_estimators': 393, 'max_depth': 14, 'min_samples_split': 12, 'min_samples_leaf': 5, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.9966251289615836.
[I 2026-05-29 15:16:02,939] Trial 3 finished with value: 0.9965543490484516 and parameters: {'n_estimators': 182, 'max_depth': 17, '

{'n_estimators': 165, 'max_depth': 18, 'min_samples_split': 8, 'min_samples_leaf': 6, 'max_features': 'sqrt'}


In [17]:
rf_model = RandomForestClassifier(
    **best_params,
    n_jobs=-1,
    random_state=42
)

rf_model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",165
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",18
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",8
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",6
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

In [18]:
# PREDICTIONS
y_proba = rf_model.predict_proba(X_test)[:, 1]

# OPTIMAL THRESHOLD (F1-based)
precision, recall, thresholds = precision_recall_curve(y_test, y_proba)

f1_scores = (2 * precision * recall) / (precision + recall + 1e-9)

best_idx = f1_scores[:-1].argmax()
best_threshold = thresholds[best_idx]

y_pred = (y_proba >= best_threshold).astype(int)

# EVALUATION
print("\nBest threshold:", best_threshold)

print("\n=== CONFUSION MATRIX ===")
print(confusion_matrix(y_test, y_pred))

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred))

print("\nROC-AUC:", roc_auc_score(y_test, y_proba))
print("PR-AUC:", average_precision_score(y_test, y_proba))


Best threshold: 0.6637659239684912

=== CONFUSION MATRIX ===
[[ 73266   9968]
 [   715 346661]]

=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

           0       0.99      0.88      0.93     83234
           1       0.97      1.00      0.98    347376

    accuracy                           0.98    430610
   macro avg       0.98      0.94      0.96    430610
weighted avg       0.98      0.98      0.97    430610


ROC-AUC: 0.9876278691984458
PR-AUC: 0.99669400014491


In [ ]:
# FEATURE IMPORTANCE
importances = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

print("\nTOP 20 FEATURES:")
print(importances.head(20))

#### 3) Modelos No Supervisados

**Isolation Forest**

In [21]:
cv = KFold(n_splits=3, shuffle=True, random_state=42)

def objective(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 120),
        "max_samples": trial.suggest_categorical("max_samples", ["auto", 0.7]),
        "max_features": trial.suggest_float("max_features", 0.8, 1.0),
        "contamination": trial.suggest_float("contamination", 0.01, 0.1),
        "random_state": 42,
        "n_jobs": -1
    }

    X = X_train_N  # numpy array

    stability_scores = []

    for tr_idx, val_idx in cv.split(X):

        X_tr = X[tr_idx]
        X_val = X[val_idx]

        model = IsolationForest(**params)
        model.fit(X_tr)

        scores = model.decision_function(X_val)

        stability_scores.append(np.std(scores))

    return np.mean(stability_scores)


study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=5)

print(study.best_params)

[I 2026-05-29 17:39:01,088] A new study created in memory with name: no-name-adef7921-8bfd-4e23-b907-aa8204fe403d
[I 2026-05-29 17:44:58,385] Trial 0 finished with value: 0.059756175576786856 and parameters: {'n_estimators': 73, 'max_samples': 'auto', 'max_features': 0.9890479777069473, 'contamination': 0.02061924357229377}. Best is trial 0 with value: 0.059756175576786856.
[I 2026-05-29 17:53:16,321] Trial 1 finished with value: 0.03090827334043596 and parameters: {'n_estimators': 93, 'max_samples': 0.7, 'max_features': 0.9042145204811527, 'contamination': 0.012560580087778528}. Best is trial 1 with value: 0.03090827334043596.
[I 2026-05-29 18:00:25,795] Trial 2 finished with value: 0.03148885114253607 and parameters: {'n_estimators': 70, 'max_samples': 0.7, 'max_features': 0.9633931465247293, 'contamination': 0.0691778115416825}. Best is trial 1 with value: 0.03090827334043596.
[I 2026-05-29 18:07:02,144] Trial 3 finished with value: 0.06002337414110448 and parameters: {'n_estimators

{'n_estimators': 93, 'max_samples': 0.7, 'max_features': 0.9042145204811527, 'contamination': 0.012560580087778528}


In [22]:
best_params = study.best_params

if_model = IsolationForest(
    **best_params,

    random_state=42,
    n_jobs=-1
)

# TRAIN (unsupervised: only X_train)
if_model.fit(X_train_N)

,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",93
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",0.7
,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",0.012560580087778528
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",0.9042145204811527
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary ` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",42
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary `... versionadded:: 0.21",False


In [23]:
# SCORES
scores = -if_model.decision_function(X_test_N)

# PR CURVE
precision, recall, thresholds = precision_recall_curve(y_test_N, scores)

# PR-AUC
pr_auc = average_precision_score(y_test_N, scores)

# BEST THRESHOLD (F1)
f1 = (2 * precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-9)
best_threshold = thresholds[np.argmax(f1)]

# PREDICTIONS
y_pred = (scores >= best_threshold).astype(int)

# EVALUATION
print("\nBest threshold:", best_threshold)

print("\n=== CONFUSION MATRIX ===")
print(confusion_matrix(y_test_N, y_pred))

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_test_N, y_pred))

print("\nROC-AUC:", roc_auc_score(y_test_N, scores))
print("PR-AUC:", pr_auc)


Best threshold: -0.07461825615051065

=== CONFUSION MATRIX ===
[[318676  28700]
 [ 96856 319313]]

=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

         0.0       0.77      0.92      0.84    347376
         1.0       0.92      0.77      0.84    416169

    accuracy                           0.84    763545
   macro avg       0.84      0.84      0.84    763545
weighted avg       0.85      0.84      0.84    763545


ROC-AUC: 0.8841060798944749
PR-AUC: 0.8966606730164193


**Autoencoder**

In [26]:
def objective(trial):

    # =========================
    # HYPERPARAMETERS
    # =========================
    latent_dim = trial.suggest_int("latent_dim", 4, 24)
    lr = trial.suggest_float("lr", 1e-4, 3e-3, log=True)

    batch_size = trial.suggest_categorical("batch_size", [128, 256])

    n1 = trial.suggest_int("n1", 32, 128)
    n2 = trial.suggest_int("n2", 16, n1)

    dropout_rate = trial.suggest_float("dropout", 0.0, 0.3)

    X = X_train_N[:5000]  # keep fast

    input_dim = X.shape[1]

    # =========================
    # MODEL (regularized AE)
    # =========================
    inp = layers.Input(shape=(input_dim,))

    x = layers.Dense(n1, activation="relu",
                     kernel_regularizer=tf.keras.regularizers.l2(1e-4))(inp)
    x = layers.Dropout(dropout_rate)(x)

    x = layers.Dense(n2, activation="relu",
                     kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x = layers.Dropout(dropout_rate)(x)

    latent = layers.Dense(latent_dim, activation="relu")(x)

    x = layers.Dense(n2, activation="relu")(latent)
    x = layers.Dense(n1, activation="relu")(x)

    out = layers.Dense(input_dim)(x)

    autoencoder = Model(inp, out)

    autoencoder.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="mae"
    )

    # =========================
    # TRAINING
    # =========================
    autoencoder.fit(
        X, X,
        epochs=10,
        batch_size=batch_size,
        validation_split=0.1,
        shuffle=True,
        verbose=0
    )

    # =========================
    # RECONSTRUCTION ERROR
    # =========================
    recon = autoencoder.predict(X, verbose=0)
    error = np.mean(np.abs(X - recon), axis=1)

    # =========================
    # UNSUPERVISED OBJECTIVE
    # (maximize separability proxy)
    # =========================
    spread = np.percentile(error, 95) - np.percentile(error, 5)
    mean_error = np.mean(error)

    # combine: encourage structure + avoid collapse
    score = spread + 0.1 * mean_error

    return score

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)
best = study.best_params
print(best)

[I 2026-05-29 18:18:21,260] A new study created in memory with name: no-name-2b77d7c7-7fe4-4689-a18b-af24e273bc9f
E0000 00:00:1780071501.457105   41218 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_UNKNOWN: unknown error
I0000 00:00:1780071501.458672   41218 cuda_diagnostics.cc:171] verbose logging is disabled. Rerun with verbose logging (usually --v=1 or --vmodule=cuda_diagnostics=1) to get more diagnostic output from this module
I0000 00:00:1780071501.458679   41218 cuda_diagnostics.cc:176] retrieving CUDA diagnostic information for host: debian
I0000 00:00:1780071501.458684   41218 cuda_diagnostics.cc:183] hostname: debian
I0000 00:00:1780071501.458825   41218 cuda_diagnostics.cc:190] libcuda reported version is: 535.261.3
I0000 00:00:1780071501.458853   41218 cuda_diagnostics.cc:194] kernel reported version is: 535.261.3
I0000 00:00:1780071501.458856   41218 cuda_diagnostics.cc:284] kernel version seems to match DSO: 535.261.3
[

{'latent_dim': 21, 'lr': 0.00017930199059425827, 'batch_size': 256, 'n1': 56, 'n2': 54, 'dropout': 0.24043288407991234}


In [27]:
best = study.best_params

inp = layers.Input(shape=(X_train_N.shape[1],))

x = layers.Dense(
    best["n1"], activation="relu",
    kernel_regularizer=tf.keras.regularizers.l2(1e-4)
)(inp)

x = layers.Dense(
    best["n2"], activation="relu",
    kernel_regularizer=tf.keras.regularizers.l2(1e-4)
)(x)

latent = layers.Dense(best["latent_dim"], activation="relu")(x)

x = layers.Dense(
    best["n2"], activation="relu",
    kernel_regularizer=tf.keras.regularizers.l2(1e-4)
)(latent)

x = layers.Dense(
    best["n1"], activation="relu",
    kernel_regularizer=tf.keras.regularizers.l2(1e-4)
)(x)

out = layers.Dense(X_train_N.shape[1])(x)

autoencoder = Model(inp, out)

autoencoder.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=best["lr"]),
    loss="mae"
)

# =========================
# EARLY STOPPING (IMPORTANT)
# =========================
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="loss",
    patience=5,
    restore_best_weights=True
)

# =========================
# TRAINING
# =========================
autoencoder.fit(
    X_train_N,
    X_train_N,
    epochs=30,
    batch_size=best["batch_size"],
    shuffle=True,
    verbose=1,
    callbacks=[early_stop]
)

Epoch 1/30
5428/5428 ━━━━━━━━━━━━━━━━━━━━ 13s 2ms/step - loss: 0.4884
Epoch 2/30
5428/5428 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step - loss: 0.3618
Epoch 3/30
5428/5428 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step - loss: 0.3206
Epoch 4/30
5428/5428 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step - loss: 0.3234
Epoch 5/30
5428/5428 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step - loss: 0.3005
Epoch 6/30
5428/5428 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - loss: 0.2955
Epoch 7/30
5428/5428 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - loss: 0.2904
Epoch 8/30
5428/5428 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step - loss: 0.2881
Epoch 9/30
5428/5428 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - loss: 0.2751
Epoch 10/30
5428/5428 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - loss: 0.2960
Epoch 11/30
5428/5428 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - loss: 0.2646
Epoch 12/30
5428/5428 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - loss: 0.2673
Epoch 13/30
5428/5428 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step - loss: 0.2642
Epoch 14/30
5428/5428 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - loss: 0.2547
Epoch 15/30
542

In [28]:
# =====================================================
# SCORES (RECONSTRUCTION ERROR)
# =====================================================
recon = autoencoder.predict(X_test_N, verbose=0)
scores = np.mean(np.abs(X_test_N - recon), axis=1)

# =====================================================
# THRESHOLD (PR OPTIMIZED)
# =====================================================
precision, recall, thresholds = precision_recall_curve(y_test_N, scores)
pr_auc = average_precision_score(y_test_N, scores)

# F1 aligned with thresholds (precision/recall are length N, thresholds N-1)
f1 = (2 * precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-9)

best_idx = np.argmax(f1)
best_threshold = thresholds[best_idx]

print("\nBest threshold:", best_threshold)

# =====================================================
# PREDICTIONS
# =====================================================
y_pred = (scores >= best_threshold).astype(int)

# =====================================================
# EVALUATION
# =====================================================
print("\n=== CONFUSION MATRIX ===")
print(confusion_matrix(y_test_N, y_pred))

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_test_N, y_pred))

print("\nROC-AUC:", roc_auc_score(y_test_N, scores))
print("PR-AUC:", pr_auc)


Best threshold: 0.11789231879834007

=== CONFUSION MATRIX ===
[[304484  42892]
 [ 58762 357407]]

=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

         0.0       0.84      0.88      0.86    347376
         1.0       0.89      0.86      0.88    416169

    accuracy                           0.87    763545
   macro avg       0.87      0.87      0.87    763545
weighted avg       0.87      0.87      0.87    763545


ROC-AUC: 0.874914509566335
PR-AUC: 0.8387774906510324
